In [ ]:
import numpy as np
import pandas as pd
import torch
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import LabelEncoder
from torch import nn

In [ ]:
df = pd.read_csv('https://raw.githubusercontent.com/gscdit/Breast-Cancer-Detection/refs/heads/master/data.csv')
df.head()

,id,diagnosis,radius_mean,texture_mean,perimeter_mean,area_mean,smoothness_mean,compactness_mean,concavity_mean,concave points_mean,...,texture_worst,perimeter_worst,area_worst,smoothness_worst,compactness_worst,concavity_worst,concave points_worst,symmetry_worst,fractal_dimension_worst,Unnamed: 32
0,842302,M,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,...,17.33,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890,NaN
1,842517,M,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,...,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902,NaN
2,84300903,M,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,...,25.53,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758,NaN
3,84348301,M,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,...,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300,NaN
4,84358402,M,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,...,16.67,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678,NaN


In [ ]:
df.shape

(569, 33)

In [ ]:
df.drop(columns=['id', 'Unnamed: 32'], inplace= True)

In [ ]:
df.head()

,diagnosis,radius_mean,texture_mean,perimeter_mean,area_mean,smoothness_mean,compactness_mean,concavity_mean,concave points_mean,symmetry_mean,...,radius_worst,texture_worst,perimeter_worst,area_worst,smoothness_worst,compactness_worst,concavity_worst,concave points_worst,symmetry_worst,fractal_dimension_worst
0,M,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,0.2419,...,25.38,17.33,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890
1,M,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,0.1812,...,24.99,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902
2,M,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,0.2069,...,23.57,25.53,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758
3,M,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,0.2597,...,14.91,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300
4,M,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,0.1809,...,22.54,16.67,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(df.iloc[:, 1:], df.iloc[:, 0], test_size=0.2)

In [ ]:
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [ ]:
encoder = LabelEncoder()
y_train = encoder.fit_transform(y_train)
y_test = encoder.transform(y_test)

In [ ]:
X_train_tensor = torch.from_numpy(X_train)
X_test_tensor = torch.from_numpy(X_test)
y_train_tensor = torch.from_numpy(y_train)
y_test_tensor = torch.from_numpy(y_test)

In [ ]:
X_train_tensor.shape

torch.Size([455, 30])

In [ ]:
y_train_tensor.shape

torch.Size([455])

In [ ]:
from torch.utils.data import Dataset, DataLoader

class CustomDataset(Dataset):
  def __init__(self, features, labels):
    self.features = features
    self.labels = labels

  def __len__(self):
    return len(self.features)

  def __getitem__(self, index):
    return self.features[index], self.labels[index]

In [ ]:
train_dataset = CustomDataset(X_train_tensor, y_train_tensor)
test_dataset = CustomDataset(X_test_tensor, y_test_tensor)

In [ ]:
train_dataset[10]

(tensor([ 3.9483, -0.1928,  3.9586,  5.2164,  1.2568,  0.9384,  2.9562,  2.8783,
         -0.6233, -1.0783,  8.8121,  0.4587,  9.3367, 10.2972,  2.2969,  0.1446,
          1.0355,  0.3797,  3.1675,  0.2436,  2.4313, -1.1741,  2.4058,  2.8156,
         -0.8157, -0.6572,  0.2370,  0.6932, -1.9738, -1.6451],
        dtype=torch.float64),
 tensor(1))

In [ ]:
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=True)

In [ ]:
class MySimpleNN(nn.Module):
  def __init__(self, num_features):
    super().__init__()
    self.linear = nn.Linear(num_features, 1)
    self.sigmoid = nn.Sigmoid()

  def forward(self, features):
    out = self.linear(features)
    out = self.sigmoid(out)
    return out

In [ ]:
learning_rate = 0.1
epochs = 25

In [ ]:
loss_function = nn.BCELoss()

In [ ]:
# create model
model = MySimpleNN(X_train_tensor.shape[1])

# Ensure input and target tensors are float32 to match model weights
X_train_tensor_f32 = X_train_tensor.float()
y_train_tensor_f32 = y_train_tensor.float()

optimizer = torch.optim.SGD(model.parameters(), lr = learning_rate)

# define loop
for epoch in range(epochs):
  for batch_features, batch_labels in train_loader: # New code

    # forward pass
    y_pred = model(batch_features.float())

    # loss calculate
    loss = loss_function(y_pred, batch_labels.float().view(-1,1))

    # zero gradients
    optimizer.zero_grad()

    # backward pass
    loss.backward()

    # parameters update
    optimizer.step()

    # manual way of optimizing gradients
    # parameters update
      #with torch.no_grad():
        #model.linear.weight -= learning_rate * model.linear.weight.grad
        #model.linear.bias -= learning_rate * model.linear.bias.grad
    # making the value of gradients zero becoz it gets accumulated after every epoch
      # zero gradients
        # model.linear.weight.grad.zero_()
        # model.linear.bias.grad.zero_()

    # print loss in each epoch
    print(f'Epoch: {epoch + 1}, Loss: {loss.item()}')

Epoch: 1, Loss: 0.8986955881118774
Epoch: 1, Loss: 0.6470350027084351
Epoch: 1, Loss: 0.4525948166847229
Epoch: 1, Loss: 0.38566288352012634
Epoch: 1, Loss: 0.41623756289482117
Epoch: 1, Loss: 0.34193918108940125
Epoch: 1, Loss: 0.33018410205841064
Epoch: 1, Loss: 0.2810809016227722
Epoch: 1, Loss: 0.24642477929592133
Epoch: 1, Loss: 0.2678336203098297
Epoch: 1, Loss: 0.30362898111343384
Epoch: 1, Loss: 0.31511831283569336
Epoch: 1, Loss: 0.22301334142684937
Epoch: 1, Loss: 0.2664147913455963
Epoch: 1, Loss: 0.27379822731018066
Epoch: 2, Loss: 0.17407067120075226
Epoch: 2, Loss: 0.2434849739074707
Epoch: 2, Loss: 0.22027631103992462
Epoch: 2, Loss: 0.15776598453521729
Epoch: 2, Loss: 0.13242687284946442
Epoch: 2, Loss: 0.2596694827079773
Epoch: 2, Loss: 0.1838100105524063
Epoch: 2, Loss: 0.26243844628334045
Epoch: 2, Loss: 0.16682395339012146
Epoch: 2, Loss: 0.2097126990556717
Epoch: 2, Loss: 0.22774749994277954
Epoch: 2, Loss: 0.1960890293121338
Epoch: 2, Loss: 0.162048801779747
Epoch

In [ ]:
# model evaluation
with torch.no_grad():
  for batch_features, batch_labels in test_loader:
   # Ensure X_test_tensor is float32 for consistency with model weights
   X_test_tensor_f32 = X_test_tensor.float()
   y_pred = model.forward(batch_features.float())
   y_pred = (y_pred > 0.9).float()
   # Ensure y_test_tensor is float32 for comparison
   accuracy = (y_pred == batch_labels).float().mean()
   print(f'Accuracy: {accuracy.item()}')

Accuracy: 0.55859375
Accuracy: 0.55859375
Accuracy: 0.58203125
Accuracy: 0.5740740895271301
